# embebidos-3 — autolabel_vastai

**Pipeline 100% automático de auto-labeling** en una instancia Vast.ai corta (~30 min). Corre Autodistill + Grounding DINO sobre un batch de imágenes alojado en HF Hub, sube las labels resultantes a otro repo HF, valida gate de calidad y se auto-destruye.

**Patrón análogo a `train_track_b_yolov8.ipynb`** pero para inference (no training):
- Sin `CommitScheduler` (job dura ~5 min, no horas).
- Sin gates ONNX/Polygraphy.
- Auto-destroy condicional al gate de calidad (no incondicional).
- Sin freeze del backbone ni hiperparámetros augmentation (es inference puro).

**Provisioning recomendado** (desde la PC del usuario):
```bash
# Buscar oferta RTX 3060 o T4 (Grounding DINO tiny necesita ~4 GB)
vastai search offers 'gpu_name=RTX_3060 gpu_ram>=10 reliability>0.95 rentable=true' \
    --order='dph_total+' --limit 3

# Provisionar con notebook ejecutado headless via nbconvert
vastai create instance OFFER_ID \
    --image pytorch/pytorch:2.4.0-cuda12.4-cudnn9-runtime \
    --disk 20 --ssh --direct \
    --env '-e HF_TOKEN=$HF_TOKEN -e BATCH=batch1' \
    --onstart-cmd "git clone <REPO> /workspace/embebidos-3 && \
        cd /workspace/embebidos-3 && \
        pip install -q -U jupyter nbformat nbclient && \
        jupyter nbconvert --execute --inplace \
            --ExecutePreprocessor.timeout=1800 \
            notebooks/autolabel_vastai.ipynb"
```

**Inputs:** repo HF dataset `mitgar14/embebidos3-raw-batches/<batch>/` con imágenes JPG crudas.

**Outputs:** repo HF dataset `mitgar14/embebidos3-labels/<batch>/` con `train/{images,labels}` formato YOLO + `data.yaml`.


## 1) Env setup

Instala deps en la instancia Vast.ai. Idempotente.

In [ ]:
import subprocess, sys

# Orden importante: deps de sistema antes de autodistill (transitivas no declaradas)
REQ = [
    "opencv-python-headless>=4.8",  # antes de autodistill: evita libxcb1/libgl1 del opencv full
    "transformers<5",                 # autodistill rompe con transformers 5+ (AutoTokenizer)
    "scikit-learn",                   # transitiva no declarada por autodistill
    "roboflow",                       # transitiva no declarada por autodistill
    "timm>=1.0",
    "accelerate",
    "autodistill>=0.1.20",
    "autodistill-grounding-dino>=0.1.0",
    "supervision>=0.21",
    "huggingface_hub>=0.24",
    "pillow>=10",
    "pyyaml>=6",
    "vastai",  # para auto-destroy desde dentro
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + REQ, check=True)
print("[OK] deps instaladas")


## 2) Config

Parámetros del job: ontology, conf, batch source/destino HF, modelo Grounding DINO.

**Por qué los prompts de la ontology son descriptivos** (no solo "plastic"): el preentrenamiento de Grounding DINO incluye `"plastic bottle"` y `"plastic container"` como categorías frecuentes; el prompt corto `"plastic"` confunde con tela o superficies plásticas no-objeto.

**Por qué `model_type=tiny`**: cabe en 6-8 GB VRAM (~3.4 GB peak) y la calidad para objetos comunes (botellas, papel, vidrio) es comparable a base; base sólo gana en clases ambiguas o visualmente similares (Voxel51 whitepaper junio 2025).

In [ ]:
import os
from pathlib import Path

# === Configurable desde env vars (Vast.ai --env) ===========================
BATCH = os.environ.get("BATCH", "batch1")  # batch1, batch2, ...
INPUT_REPO = os.environ.get("INPUT_REPO", "mitgar14/embebidos3-raw-batches")
OUTPUT_REPO = os.environ.get("OUTPUT_REPO", "mitgar14/embebidos3-labels")
CONF = float(os.environ.get("CONF", "0.25"))
MODEL_TYPE = os.environ.get("MODEL_TYPE", "tiny")  # tiny | base
MIN_RECALL_RATIO = float(os.environ.get("MIN_RECALL_RATIO", "0.85"))
AUTO_DESTROY = os.environ.get("AUTO_DESTROY", "true").lower() in ("true", "1", "yes")

# === Ontology: prompts descriptivos para clases YOLO ======================
ONTOLOGY = {
    "plastic bottle or plastic container": "plastic",
    "paper or cardboard": "paper",
    "glass bottle or glass jar": "glass",
}

# === Paths locales en la instancia =========================================
WORK_DIR = Path("/workspace")
INPUTS_DIR = WORK_DIR / "inputs" / BATCH
OUTPUT_DIR = WORK_DIR / "autodistill-output"

print(f"[CONFIG] batch={BATCH}")
print(f"[CONFIG] input={INPUT_REPO}/{BATCH}/")
print(f"[CONFIG] output={OUTPUT_REPO}/{BATCH}/")
print(f"[CONFIG] ontology={ONTOLOGY}")
print(f"[CONFIG] conf={CONF}  model_type={MODEL_TYPE}  min_recall_ratio={MIN_RECALL_RATIO}")
print(f"[CONFIG] auto_destroy={AUTO_DESTROY}")


## 3) Pre-flight

Validaciones que detienen el job antes de gastar tiempo de GPU:
- CUDA disponible y suficiente VRAM
- HF_TOKEN válido
- INPUT_REPO accesible
- Espacio en disco suficiente

In [ ]:
import shutil
import torch
from huggingface_hub import HfApi

# --- CUDA ----------------------------------------------------------------
assert torch.cuda.is_available(), "CUDA NO disponible: revisar driver host"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"[OK] GPU={gpu_name}  VRAM={vram_gb:.1f} GB")
if vram_gb < 6.5:
    print("[WARN] VRAM < 6.5 GB, considerar model_type=tiny obligatorio")

# --- HF token y acceso al repo --------------------------------------------
hf_token = os.environ.get("HF_TOKEN")
assert hf_token, "HF_TOKEN ausente. Inyectar via vastai create --env"
api = HfApi(token=hf_token)
me = api.whoami()
print(f"[OK] HF user={me.get('name')}")

info = api.dataset_info(INPUT_REPO)
print(f"[OK] Acceso a {INPUT_REPO} (private={info.private}, last_modified={info.lastModified})")

# --- Espacio en disco -----------------------------------------------------
free_gb = shutil.disk_usage("/workspace").free / 1e9
print(f"[OK] Disco libre /workspace: {free_gb:.1f} GB")
assert free_gb > 5, "Disco insuficiente (<5 GB libre)"

# --- vastai (para auto-destroy) -------------------------------------------
if AUTO_DESTROY:
    container_api_key = os.environ.get("CONTAINER_API_KEY")
    if not container_api_key:
        print("[WARN] CONTAINER_API_KEY no inyectada: auto-destroy desactivado")
        AUTO_DESTROY = False
    else:
        print("[OK] CONTAINER_API_KEY presente (auto-destroy armado)")


## 4) Descargar imágenes desde HF dataset

Usa `snapshot_download` con filtro `allow_patterns` para bajar solo el batch deseado, no todo el repo.

In [ ]:
from huggingface_hub import snapshot_download

snap = snapshot_download(
    repo_id=INPUT_REPO,
    repo_type="dataset",
    allow_patterns=[f"{BATCH}/*.jpg", f"{BATCH}/*.jpeg", f"{BATCH}/*.png"],
    local_dir=str(WORK_DIR / "inputs_snapshot"),
    token=hf_token,
)
INPUTS_DIR = Path(snap) / BATCH
imgs = sorted(INPUTS_DIR.glob("*.jpg")) + sorted(INPUTS_DIR.glob("*.jpeg")) + sorted(INPUTS_DIR.glob("*.png"))
n_inputs = len(imgs)
assert n_inputs > 0, f"No se encontraron imagenes en {INPUTS_DIR}"
print(f"[OK] Descargadas {n_inputs} imagenes en {INPUTS_DIR}")
print("Sample:", [p.name for p in imgs[:5]])


## 5) Auto-label con Grounding DINO

Genera estructura YOLO en `OUTPUT_DIR`:
```
autodistill-output/
├── data.yaml
├── train/
│   ├── images/*.jpg
│   └── labels/*.txt
└── valid/ (vacío — Autodistill no hace split automático)
```

In [ ]:
import time
from autodistill.detection import CaptionOntology
from autodistill_grounding_dino import GroundingDINO

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
# API actual de autodistill-grounding-dino: NO acepta model_type ni conf en .label().
# Los thresholds se pasan al constructor: box_threshold (objectness) + text_threshold (CLIP score).
# text_threshold se baja ~30% relativo a box_threshold (heuristica recomendada por IDEA-Research).
print(f"[INFO] Cargando GroundingDINO box_threshold={CONF} text_threshold={CONF * 0.7:.3f}...")
base = GroundingDINO(
    ontology=CaptionOntology(ONTOLOGY),
    box_threshold=CONF,
    text_threshold=CONF * 0.7,
)
print(f"[INFO] Modelo cargado en {time.time() - t0:.1f}s. Etiquetando {n_inputs} imagenes...")

base.label(
    input_folder=str(INPUTS_DIR),
    output_folder=str(OUTPUT_DIR),
    extension=".jpg",  # autodistill no acepta multiples extensiones; convertir .png si fuese necesario
)
dt = time.time() - t0
print(f"[OK] Auto-label completado en {dt:.1f}s ({dt/n_inputs:.2f}s/img)")


## 6) Gate de calidad

Verifica que al menos `MIN_RECALL_RATIO` (default 85%) de las imágenes tengan al menos 1 bbox. Si no, **aborta sin destruir** la instancia para inspección manual.

Reporta también distribución por clase para detectar sesgos del modelo (si Grounding DINO subestima papel/vidrio, el ratio será menor para esas clases).

In [ ]:
from collections import Counter
import yaml

# Autodistill hace split train/valid automatico. Contar AMBOS para recall real.
all_label_files = sorted((OUTPUT_DIR / "train" / "labels").glob("*.txt")) + \
                  sorted((OUTPUT_DIR / "valid" / "labels").glob("*.txt"))
non_empty = [p for p in all_label_files if p.stat().st_size > 0]
n_labels = len(non_empty)
ratio = n_labels / max(n_inputs, 1)

# Distribucion por clase (train + valid)
class_counts: Counter = Counter()
for p in non_empty:
    for line in p.read_text().splitlines():
        if not line.strip():
            continue
        cls_idx = int(line.split()[0])
        class_counts[cls_idx] += 1

data_yaml_path = OUTPUT_DIR / "data.yaml"
if data_yaml_path.exists():
    names = yaml.safe_load(data_yaml_path.read_text()).get("names", [])
else:
    names = sorted(set(ONTOLOGY.values()))

n_train = len(list((OUTPUT_DIR / "train" / "labels").glob("*.txt")))
n_valid = len(list((OUTPUT_DIR / "valid" / "labels").glob("*.txt")))
print(f"[GATE] Split autodistill: train={n_train}, valid={n_valid}")
print(f"[GATE] Imagenes con >= 1 bbox: {n_labels}/{n_inputs} = {ratio:.1%} (umbral {MIN_RECALL_RATIO:.0%})")
print(f"[GATE] Total bboxes: {sum(class_counts.values())}")
for cls_idx, count in sorted(class_counts.items()):
    name = names[cls_idx] if cls_idx < len(names) else f"class_{cls_idx}"
    print(f"  {name}: {count}")

if ratio < MIN_RECALL_RATIO:
    raise RuntimeError(
        f"GATE FAILED: ratio {ratio:.1%} < {MIN_RECALL_RATIO:.0%}. "
        "Revisar conf threshold o ontology. No se destruye la instancia."
    )
print("[OK] Gate de calidad pasado")


## 7) Upload labels a HF Hub

Sube el folder completo (`train/images/`, `train/labels/`, `data.yaml`) al repo destino bajo el subpath `<batch>/`. Si el repo no existe, lo crea privado.

In [ ]:
from huggingface_hub import create_repo, upload_folder

create_repo(
    repo_id=OUTPUT_REPO,
    token=hf_token,
    repo_type="dataset",
    private=True,
    exist_ok=True,
)

url = upload_folder(
    folder_path=str(OUTPUT_DIR),
    path_in_repo=BATCH,
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    token=hf_token,
    commit_message=(
        f"auto-label {BATCH}: {n_labels}/{n_inputs} imgs, "
        f"{sum(class_counts.values())} bboxes (gd-{MODEL_TYPE}, conf={CONF})"
    ),
)
print(f"[DONE] Upload OK: https://huggingface.co/datasets/{OUTPUT_REPO}/tree/main/{BATCH}")


## 8) Auto-destroy

Destruye la propia instancia usando `CONTAINER_API_KEY` (inyectada por Vast.ai con scope limitado a esta única instancia). Si esta celda falla, el `onstart_cmd` que orquesta el notebook también puede destruir como fallback.

In [ ]:
import subprocess

if AUTO_DESTROY and (container_api_key := os.environ.get("CONTAINER_API_KEY")):
    print("[INFO] Auto-destroy en 5s... (Ctrl+C para abortar)")
    time.sleep(5)
    result = subprocess.run(
        ["vastai", "destroy", "instance", container_api_key],
        capture_output=True,
        text=True,
    )
    print("[vastai stdout]", result.stdout)
    print("[vastai stderr]", result.stderr)
else:
    print("[INFO] AUTO_DESTROY desactivado o CONTAINER_API_KEY ausente. "
          "Destruir manualmente: vastai destroy instance <ID>")
